In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

# Configuration
DAYS_BACK = 365
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=DAYS_BACK)

# Top 10 cryptocurrencies by market cap
CRYPTO_IDS = [
    "bitcoin", "ethereum", "tether", "binancecoin", "solana",
    "ripple", "usd-coin", "cardano", "dogecoin", "tron"
]

# 10 major forex pairs (USD base)
FOREX_PAIRS = [
    ("USD", "EUR"),
    ("USD", "GBP"),
    ("USD", "JPY"),
    ("USD", "CHF"),
    ("USD", "CAD"),
    ("USD", "AUD"),
    ("USD", "NZD"),
    ("USD", "CNY"),
    ("USD", "INR"),
    ("USD", "SGD")
]

print(f"✅ Configuration loaded")
print(f"✅ Date range: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")
print(f"✅ Tracking {len(CRYPTO_IDS)} cryptocurrencies")
print(f"✅ Tracking {len(FOREX_PAIRS)} forex pairs")

In [0]:
def fetch_crypto_historical(crypto_id: str, days: int = 365, max_retries: int = 3) -> pd.DataFrame:
    """
    Fetch historical crypto data from CoinGecko API.
    Returns DataFrame with daily OHLCV data.
    """
    url = f"https://api.coingecko.com/api/v3/coins/{crypto_id}/market_chart"
    params = {
        "vs_currency": "usd",
        "days": days,
        "interval": "daily"
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            
            # Extract prices, market caps, and volumes
            prices = data.get("prices", [])
            market_caps = data.get("market_caps", [])
            volumes = data.get("total_volumes", [])
            
            # Convert to DataFrame
            df = pd.DataFrame({
                "timestamp": [p[0] for p in prices],
                "price_usd": [p[1] for p in prices],
                "market_cap": [m[1] for m in market_caps],
                "volume_24h": [v[1] for v in volumes]
            })
            
            # Convert timestamp to date
            df["date"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
            df["crypto_id"] = crypto_id
            df["fetch_timestamp"] = datetime.now()
            
            print(f"✅ {crypto_id}: {len(df)} days fetched")
            return df
            
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429:  # Rate limit
                wait_time = 15 * (attempt + 1)  # Exponential backoff: 15s, 30s, 45s
                print(f"⚠️ Rate limited on {crypto_id}. Waiting {wait_time}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"❌ HTTP Error fetching {crypto_id}: {e}")
                return pd.DataFrame()
        except Exception as e:
            print(f"❌ Error fetching {crypto_id}: {e}")
            return pd.DataFrame()
    
    print(f"❌ Failed to fetch {crypto_id} after {max_retries} attempts")
    return pd.DataFrame()

# Fetch all cryptocurrencies
print(f"Fetching {DAYS_BACK} days of data for {len(CRYPTO_IDS)} cryptocurrencies...\n")
all_crypto_data = []

for i, crypto_id in enumerate(CRYPTO_IDS):
    print(f"[{i+1}/{len(CRYPTO_IDS)}] Fetching {crypto_id}...")
    df = fetch_crypto_historical(crypto_id, DAYS_BACK)
    
    if not df.empty:
        all_crypto_data.append(df)
    
    # Rate limiting - CoinGecko free tier: ~10-30 calls/min
    if i < len(CRYPTO_IDS) - 1:
        time.sleep(7)  # 7 second delay = ~8.5 calls/min (conservative)

# Combine all data
if all_crypto_data:
    crypto_df = pd.concat(all_crypto_data, ignore_index=True)
    print(f"\n✅ Total crypto records: {len(crypto_df)}")
    print(f"✅ Date range: {crypto_df['date'].min()} to {crypto_df['date'].max()}")
    print(f"✅ Unique cryptocurrencies: {crypto_df['crypto_id'].nunique()}")
else:
    print("❌ No crypto data fetched")
    crypto_df = pd.DataFrame()

In [0]:
# Save crypto data to Bronze layer
if not crypto_df.empty:
    # Convert pandas DataFrame to Spark DataFrame
    spark_crypto_df = spark.createDataFrame(crypto_df)
    
    # Add metadata columns
    from pyspark.sql.functions import lit, current_timestamp
    spark_crypto_df = spark_crypto_df \
        .withColumn("source", lit("coingecko")) \
        .withColumn("ingestion_timestamp", current_timestamp())
    
    print("\nSaving crypto data to Bronze layer...")
    print(f"Total records: {spark_crypto_df.count()}")
    
    # Write to Delta table
    spark_crypto_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("crypto_id") \
        .saveAsTable("financial_market.bronze.crypto")
    
    print("\n✅ Crypto data saved to: financial_market.bronze.crypto")
    print(f"✅ Partitioned by: crypto_id")
    
    # Verify
    verify_df = spark.sql("""
        SELECT 
            crypto_id,
            COUNT(*) as record_count,
            MIN(date) as earliest_date,
            MAX(date) as latest_date
        FROM financial_market.bronze.crypto
        GROUP BY crypto_id
        ORDER BY crypto_id
    """)
    display(verify_df)
else:
    print("❌ No crypto data to save")

In [0]:
def fetch_forex_historical(base: str, target: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch historical forex rates from Frankfurter API.
    Returns DataFrame with daily exchange rates.
    """
    url = f"https://api.frankfurter.app/{start_date}..{end_date}"
    params = {
        "from": base,
        "to": target
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Extract rates by date
        rates_data = data.get("rates", {})
        
        rows = []
        for date_str, rate_obj in rates_data.items():
            rows.append({
                "date": date_str,
                "base_currency": base,
                "target_currency": target,
                "exchange_rate": rate_obj.get(target, None)
            })
        
        df = pd.DataFrame(rows)
        df["pair"] = f"{base}/{target}"
        df["fetch_timestamp"] = datetime.now()
        
        print(f"✅ {base}/{target}: {len(df)} days fetched")
        return df
        
    except Exception as e:
        print(f"❌ Error fetching {base}/{target}: {e}")
        return pd.DataFrame()

# Fetch all forex pairs
start_str = START_DATE.strftime("%Y-%m-%d")
end_str = END_DATE.strftime("%Y-%m-%d")

print(f"Fetching forex data from {start_str} to {end_str}...\n")
all_forex_data = []

for i, (base, target) in enumerate(FOREX_PAIRS):
    print(f"[{i+1}/{len(FOREX_PAIRS)}] Fetching {base}/{target}...")
    df = fetch_forex_historical(base, target, start_str, end_str)
    
    if not df.empty:
        all_forex_data.append(df)
    
    # Rate limiting
    if i < len(FOREX_PAIRS) - 1:
        time.sleep(1)  # 1 second delay

# Combine all data
if all_forex_data:
    forex_df = pd.concat(all_forex_data, ignore_index=True)
    print(f"\n✅ Total forex records: {len(forex_df)}")
    print(f"✅ Date range: {forex_df['date'].min()} to {forex_df['date'].max()}")
    print(f"✅ Unique pairs: {forex_df['pair'].nunique()}")
else:
    print("❌ No forex data fetched")
    forex_df = pd.DataFrame()

In [0]:
# Save forex data to Bronze layer
if not forex_df.empty:
    # Convert pandas DataFrame to Spark DataFrame
    spark_forex_df = spark.createDataFrame(forex_df)
    
    # Add metadata columns
    from pyspark.sql.functions import lit, current_timestamp
    spark_forex_df = spark_forex_df \
        .withColumn("source", lit("frankfurter")) \
        .withColumn("ingestion_timestamp", current_timestamp())
    
    print("\nSaving forex data to Bronze layer...")
    print(f"Total records: {spark_forex_df.count()}")
    
    # Write to Delta table
    spark_forex_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("pair") \
        .saveAsTable("financial_market.bronze.forex")
    
    print("\n✅ Forex data saved to: financial_market.bronze.forex")
    print(f"✅ Partitioned by: pair")
    
    # Verify
    verify_df = spark.sql("""
        SELECT 
            pair,
            base_currency,
            target_currency,
            COUNT(*) as record_count,
            MIN(date) as earliest_date,
            MAX(date) as latest_date
        FROM financial_market.bronze.forex
        GROUP BY pair, base_currency, target_currency
        ORDER BY pair
    """)
    display(verify_df)
else:
    print("❌ No forex data to save")

In [0]:
# Final verification - total volume
print("="*60)
print("FINAL DATA INGESTION SUMMARY")
print("="*60)

# Crypto summary
crypto_count = spark.sql("SELECT COUNT(*) as cnt FROM financial_market.bronze.crypto").collect()[0]['cnt']
crypto_assets = spark.sql("SELECT COUNT(DISTINCT crypto_id) as cnt FROM financial_market.bronze.crypto").collect()[0]['cnt']
crypto_dates = spark.sql("""
    SELECT 
        MIN(date) as min_date,
        MAX(date) as max_date,
        COUNT(DISTINCT date) as unique_dates
    FROM financial_market.bronze.crypto
""").collect()[0]

print(f"\n✅ CRYPTO DATA:")
print(f"   Total records: {crypto_count:,}")
print(f"   Unique cryptocurrencies: {crypto_assets}")
print(f"   Date range: {crypto_dates['min_date']} to {crypto_dates['max_date']}")
print(f"   Unique dates: {crypto_dates['unique_dates']}")

# Forex summary
forex_count = spark.sql("SELECT COUNT(*) as cnt FROM financial_market.bronze.forex").collect()[0]['cnt']
forex_pairs = spark.sql("SELECT COUNT(DISTINCT pair) as cnt FROM financial_market.bronze.forex").collect()[0]['cnt']
forex_dates = spark.sql("""
    SELECT 
        MIN(date) as min_date,
        MAX(date) as max_date,
        COUNT(DISTINCT date) as unique_dates
    FROM financial_market.bronze.forex
""").collect()[0]

print(f"\n✅ FOREX DATA:")
print(f"   Total records: {forex_count:,}")
print(f"   Unique pairs: {forex_pairs}")
print(f"   Date range: {forex_dates['min_date']} to {forex_dates['max_date']}")
print(f"   Unique dates: {forex_dates['unique_dates']}")

# Grand total
total_records = crypto_count + forex_count
print(f"\n{'='*60}")
print(f"✅ GRAND TOTAL: {total_records:,} records ingested")
print(f"{'='*60}")

# Expected records calculation
expected_crypto = len(CRYPTO_IDS) * DAYS_BACK
expected_forex = len(FOREX_PAIRS) * DAYS_BACK
expected_total = expected_crypto + expected_forex

print(f"\nExpected vs Actual:")
print(f"   Expected: {expected_total:,} records ({expected_crypto:,} crypto + {expected_forex:,} forex)")
print(f"   Actual:   {total_records:,} records ({crypto_count:,} crypto + {forex_count:,} forex)")
print(f"   Coverage: {(total_records/expected_total)*100:.1f}%")